# Boundary conditions

The index space is periodic by default, so an edge with no condition wraps
([`02_ode`](02_ode.ipynb) states the rule).

`OrdinaryDifferentialSystem.from_sympy` takes one right-hand side for the whole grid. A condition
overrides that right-hand side on part of the grid, and you pass conditions through `conditions`.

A condition names the region it applies on, and constrains one of two things for the sites in that
region:

- the **state**, by holding $u$ at a given value;
- the **derivative**, by setting $du/dt$ to an expression you write.

Every site outside the region keeps the original right-hand side. Each condition becomes its own
branch of the assembled equation, which `equation.cases` lists.

In [1]:
import sympy as sp

from lin_ode import (
    Box,
    DerivativeCondition,
    DirichletCondition,
    Facet,
    OrdinaryDifferentialSystem,
)

N = 10
h = sp.Symbol("h")
u = sp.IndexedBase("u")
i = sp.Idx("i", (0, N - 1))

diffusion = (u[i - 1] - 2 * u[i] + u[i + 1]) / h**2

## The periodic default

Index arithmetic on the state variable is wrapped in `Mod`, one modulus per axis, so a stencil
that reaches past an end comes back at the other. Substituting a concrete site shows the wrap
resolving by itself: `u[i - 1]` at $i = 0$ reads `u[N - 1]`, and `u[i + 1]` at $i = N - 1$ reads
`u[0]`.

In [2]:
wrapped = OrdinaryDifferentialSystem.from_sympy(
    (u[i + 1] - u[i - 1]) / (2 * h), u, (i,)
)
print("assembled ", wrapped.equation.expr)
for site in (0, 3, N - 1):
    print(f"at i = {site:<3}", wrapped.equation.expr.subs(i, site))

assembled  (u[Mod(i + 1, 10)] - u[Mod(i + 9, 10)])/(2*h)
at i = 0   (u[1] - u[9])/(2*h)
at i = 3   (-u[2] + u[4])/(2*h)
at i = 9   (u[0] - u[8])/(2*h)


A negative literal counts from the end, as a Python index does: `u[-1]` is the last site,
`u[N - 1]`.

In [3]:
print("u[-1] ->", OrdinaryDifferentialSystem.from_sympy(u[i] * u[-1], u, (i,)).equation.expr)
print("u[-2] ->", OrdinaryDifferentialSystem.from_sympy(u[i] * u[-2], u, (i,)).equation.expr)

u[-1] -> u[9]*u[i]
u[-2] -> u[8]*u[i]


## Regions

Every condition names the region it applies on, and a region is a condition on the indices.
`Facet.low(index)` and `Facet.high(index)` are the two ends of that axis: a point in 1D, an edge
in 2D, a face in 3D. `Box` restricts each axis to a range, and an axis left out of the range spans
the whole grid. Two pinned regions combine with `&`, which is how a corner is built.

A `Box` is a volume, and its `.boundary` is the shell around it: the region whose condition is the
union of the box's walls.

The two indices below are the grid the 2D example later on runs on, because a corner needs two
axes.

In [4]:
Nx, Ny = 6, 5
p, q = sp.Idx("p", (0, Nx - 1)), sp.Idx("q", (0, Ny - 1))

print("Facet.low(p)                ", Facet.low(p).condition)
print("Facet.high(q)               ", Facet.high(q).condition)
print("Facet.low(p) & Facet.low(q) ", (Facet.low(p) & Facet.low(q)).condition)
print("Box({p: (2, 3)})            ", Box({p: (2, 3)}).condition)
print("Box({p: (2, 3), q: (1, 2)}) ", Box({p: (2, 3), q: (1, 2)}).condition)
print("Box({i: (2, 5)}).boundary   ", Box({i: (2, 5)}).boundary.condition)

Facet.low(p)                 Eq(p, 0)
Facet.high(q)                Eq(q, 4)
Facet.low(p) & Facet.low(q)  Eq(p, 0) & Eq(q, 0)
Box({p: (2, 3)})             (p >= 2) & (p <= 3)
Box({p: (2, 3), q: (1, 2)})  (p >= 2) & (q >= 1) & (p <= 3) & (q <= 2)
Box({i: (2, 5)}).boundary    Eq(i, 2) | Eq(i, 5)


## Conditions on the state

A `DirichletCondition` holds the state at a value on its region. The value may be a constant, an
expression in the indices, or a reference to the state itself.

The assembled `equation` is $du/dt$, not $u$. Holding $u$ at a value $v$ fixes the derivative
through the chain rule,

$$\frac{d}{dt}u[k] = \sum_j \frac{\partial v}{\partial u[j]}\,f_j(u),$$

so a constant value gives a zero branch. The values themselves stay on `state_conditions`, where
the initial-state builder reads them; they never enter the equation, and the held sites stay
ordinary degrees of freedom whose rows are zero.

In [5]:
dirichlet = OrdinaryDifferentialSystem.from_sympy(
    diffusion,
    u,
    (i,),
    conditions=[
        DirichletCondition(Facet.low(i), 0),
        DirichletCondition(Facet.high(i), sp.Rational(1, 2)),
    ],
)
for held in dirichlet.state_conditions:
    print(f"held    {held.region.condition}  ->  u = {held.value}")
for value, case in dirichlet.equation.cases:
    print(f"branch  {case}  ->  du/dt = {value}")

held    Eq(i, 0)  ->  u = 0
held    Eq(i, 9)  ->  u = 1/2
branch  Eq(i, 0) | Eq(i, 9)  ->  du/dt = 0
branch  True  ->  du/dt = (u[Mod(i + 1, 10)] + u[Mod(i + 9, 10)] - 2*u[i])/h**2


The mirror $u[0]=u[1]$ is the second case: $\partial v/\partial u[1]=1$, so site 0 follows site 1.
`eliminate=True` substitutes references to the constrained site out of the remaining equations, so
the constrained sites no longer appear on any right-hand side. A value that references a site
another condition constrains is resolved either way.

In [6]:
mirror = OrdinaryDifferentialSystem.from_sympy(
    diffusion,
    u,
    (i,),
    conditions=[DirichletCondition(Facet.low(i), u[1])],
    eliminate=True,
)
print("at i = 0:", mirror.equation.expr.subs(i, 0))
print("at i = 1:", mirror.equation.expr.subs(i, 1))

at i = 0: (-u[1] + u[2])/h**2
at i = 1: (-u[1] + u[2])/h**2


### Why that branch

Read the chain rule above at $v = u[1]$. The only nonzero derivative is
$\partial v/\partial u[1] = 1$, so the branch is $f_1(u)$: the interior stencil evaluated at
the *referenced* site. The value selects which site's right-hand side is copied, rather than
something to substitute into site 0's own stencil.

Site 1 needs no branch of its own, because its plain stencil already reads `u[0]`, which the
condition aliases to `u[1]`. (That alias is the first-order Neumann discretisation. The
second-order ghost-point form $u[-1] := u[1]$ names a site outside the grid, and is not
expressible here.)

Assembled without `eliminate`, so the branch reads as derived:

In [7]:
alias = OrdinaryDifferentialSystem.from_sympy(
    diffusion, u, (i,), conditions=[DirichletCondition(Facet.low(i), u[1])]
)
branch = alias.equation.expr.subs(i, 0)
print("branch at i = 0          ", branch)
print("interior stencil at i = 1", diffusion.subs(i, 1))
print("same                     ", sp.simplify(branch - diffusion.subs(i, 1)) == 0)

branch at i = 0           (u[0] - 2*u[1] + u[2])/h**2
interior stencil at i = 1 (u[0] - 2*u[1] + u[2])/h**2
same                      True


### Advanced - A value that varies along its region, in 2D

A Dirichlet value may be an expression in the indices, so an edge can carry a profile. Three
conditions on a 2D Laplacian, assembled with `eliminate=True` so the held values are substituted
into the rows that reference them:

- the corner $(0, 0)$ held at $0$;
- the $p = 0$ edge given the profile $\sin(\pi q / (N_y - 1))$, which varies along the edge;
- the $q = 0$ edge mirrored to `u[p, 1]`, a derived, state-dependent branch.

All three are state conditions, and among conditions of the same family the list order decides:
the `Piecewise` is first-match, so the first-listed one wins where regions overlap. The corner
sits on both edges, so it is listed **first** for it to win at $(0, 0)$.

In [8]:
laplacian = (
    u[p - 1, q] + u[p + 1, q] + u[p, q - 1] + u[p, q + 1] - 4 * u[p, q]
) / h**2

ode2d = OrdinaryDifferentialSystem.from_sympy(
    laplacian,
    u,
    (p, q),
    conditions=[
        DirichletCondition(Facet.low(p) & Facet.low(q), 0),  # corner first, so it wins
        DirichletCondition(Facet.low(p), sp.sin(sp.pi * q / (Ny - 1))),
        DirichletCondition(Facet.low(q), u[p, 1]),  # mirror the q = 0 edge
    ],
    eliminate=True,
)
ode2d.equation.expr

Piecewise((0, Eq(p, 0) & (Eq(p, 0) | Eq(q, 0))), ((u[Mod(p + 1, 6), 1] + u[Mod(p + 5, 6), 1] - 3*u[p, 1] + u[p, 2])/h**2, Eq(q, 0)), ((-3*u[1, 1] + u[1, 2] + u[2, 1] + sqrt(2)/2)/h**2, Eq(p, 1) & Eq(q, 1)), ((u[1, 1] + u[1, 3] - 4*u[1, 4] + u[2, 4])/h**2, Eq(p, 1) & Eq(q, 4)), ((u[4, 1] - 3*u[5, 1] + u[5, 2] + sqrt(2)/2)/h**2, Eq(p, 5) & Eq(q, 1)), ((u[4, 4] + u[5, 1] + u[5, 3] - 4*u[5, 4])/h**2, Eq(p, 5) & Eq(q, 4)), ((sin(pi*q/4) + u[1, Mod(q + 1, 5)] + u[1, Mod(q + 4, 5)] - 4*u[1, q] + u[2, q])/h**2, Eq(p, 1)), ((sin(pi*q/4) + u[4, q] + u[5, Mod(q + 1, 5)] + u[5, Mod(q + 4, 5)] - 4*u[5, q])/h**2, Eq(p, 5)), ((u[Mod(p + 1, 6), 1] + u[Mod(p + 5, 6), 1] - 3*u[p, 1] + u[p, 2])/h**2, Eq(q, 1)), ((u[Mod(p + 1, 6), 4] + u[Mod(p + 5, 6), 4] + u[p, 1] + u[p, 3] - 4*u[p, 4])/h**2, Eq(q, 4)), ((u[Mod(p + 1, 6), q] + u[Mod(p + 5, 6), q] + u[p, Mod(q + 1, 5)] + u[p, Mod(q + 4, 5)] - 4*u[p, q])/h**2, True))

Evaluating at representative sites: the held edge and corner sites sit at $0$, and each interior
row *adjacent* to the $p = 0$ edge picks up the local boundary value, $\sin(\pi/2) = 1$ at
$q = 2$ and $\sin(3\pi/4) = \sqrt{2}/2$ at $q = 3$. The mirrored edge folds `u[p, 1]` into its
own stencil, and the unconstrained far edge $p = N_x - 1$ still wraps onto $p = 0$, so
elimination folds the profile into its row too.

In [9]:
sites = {
    "corner (0, 0)": (0, 0),
    "profile edge (0, 2), held": (0, 2),
    "row beside the profile edge (1, 2)": (1, 2),
    "row beside the profile edge (1, 3)": (1, 3),
    "mirrored edge (2, 0)": (2, 0),
    "unconstrained edge (5, 2)": (5, 2),
}
for label, (a, b) in sites.items():
    print(f"{label:<36}", ode2d.equation.expr.subs({p: a, q: b}))

corner (0, 0)                        0
profile edge (0, 2), held            0
row beside the profile edge (1, 2)   (u[1, 1] - 4*u[1, 2] + u[1, 3] + u[2, 2] + 1)/h**2
row beside the profile edge (1, 3)   (u[1, 2] - 4*u[1, 3] + u[1, 4] + u[2, 3] + sqrt(2)/2)/h**2
mirrored edge (2, 0)                 (u[1, 1] - 3*u[2, 1] + u[2, 2] + u[3, 1])/h**2
unconstrained edge (5, 2)            (u[4, 2] + u[5, 1] - 4*u[5, 2] + u[5, 3] + 1)/h**2


### Holding the shell of a box

`Box.boundary` is a region like any other, so a `DirichletCondition` on it holds $u$ on every wall
of the box at once. The box interior and everything outside keep the ambient stencil.

In [10]:
shell = OrdinaryDifferentialSystem.from_sympy(
    diffusion, u, (i,), conditions=[DirichletCondition(Box({i: (2, 5)}).boundary, 0)]
)
for site in (2, 3, 5):
    print(f"at i = {site}", shell.equation.expr.subs(i, site))

at i = 2 0
at i = 3 (u[2] - 2*u[3] + u[4])/h**2
at i = 5 0


## Conditions on the derivative

A `DerivativeCondition` sets $du/dt$ on its region instead of holding the state.

The usual case is an open end. Each edge site drops the neighbour that falls outside the grid, and
every other site keeps the original right-hand side.
[`08_end_to_end`](08_end_to_end.ipynb) builds its Part 2 equation this way.

In [11]:
open_ends = OrdinaryDifferentialSystem.from_sympy(
    diffusion,
    u,
    (i,),
    conditions=[
        DerivativeCondition(Facet.low(i), (u[i + 1] - 2 * u[i]) / h**2),
        DerivativeCondition(Facet.high(i), (u[i - 1] - 2 * u[i]) / h**2),
    ],
)
for value, case in open_ends.equation.cases:
    print(f"{case}  ->  du/dt = {value}")

Eq(i, 0)  ->  du/dt = (u[Mod(i + 1, 10)] - 2*u[i])/h**2
Eq(i, 9)  ->  du/dt = (u[Mod(i + 9, 10)] - 2*u[i])/h**2
True  ->  du/dt = (u[Mod(i + 1, 10)] + u[Mod(i + 9, 10)] - 2*u[i])/h**2


### A region with its own equation

The region need not be an end. Given a `Box`, a `DerivativeCondition` runs a different equation on
that part of the grid: a subdomain. Here a box $3 \le i \le 5$ diffuses with its own coefficient
$D$, embedded in plain diffusion everywhere else.

State conditions and derivative conditions are passed together in the one `conditions` argument,
and the assembler groups them by family.

In [12]:
D = sp.Symbol("D")
scaled = D * (u[i - 1] - 2 * u[i] + u[i + 1]) / h**2

subdomain = OrdinaryDifferentialSystem.from_sympy(
    diffusion, u, (i,), conditions=[DerivativeCondition(Box({i: (3, 5)}), scaled)]
)
for value, case in subdomain.equation.cases:
    print(f"{case}  ->  du/dt = {value}")

(i >= 3) & (i <= 5)  ->  du/dt = D*(u[Mod(i + 1, 10)] + u[Mod(i + 9, 10)] - 2*u[i])/h**2
True  ->  du/dt = (u[Mod(i + 1, 10)] + u[Mod(i + 9, 10)] - 2*u[i])/h**2


The box is an inequality, not a point, and a subdomain has no boundary of its own: its edge reads
the ambient field, coupling to its neighbours like any interior site. At $i = 3$ the subdomain
stencil reaches `u[2]`, a site outside the box.

The conditions stay on the system as `derivative_conditions`, the counterpart of
`state_conditions`.

In [13]:
for site in (1, 3, 4, 6):
    print(f"at i = {site}", subdomain.equation.expr.subs(i, site))

for rule in subdomain.derivative_conditions:
    print(f"rule    {rule.region.condition}  ->  du/dt = {rule.rhs}")

at i = 1 (u[0] - 2*u[1] + u[2])/h**2
at i = 3 D*(u[2] - 2*u[3] + u[4])/h**2
at i = 4 D*(u[3] - 2*u[4] + u[5])/h**2
at i = 6 (u[5] - 2*u[6] + u[7])/h**2
rule    (i >= 3) & (i <= 5)  ->  du/dt = D*(u[i + 1] + u[i - 1] - 2*u[i])/h**2


## Which condition wins

Two regions can name the same site, and the rule has two parts:

- **across the families**, a state condition beats a derivative condition. Holding $u$ is a
  stronger statement than prescribing $du/dt$, and the list order plays no part.
- **within one family**, the first-listed condition wins, because the `Piecewise` is first-match.
  This is the rule the 2D corner relies on above.

The first part is what settles a box edge lying on a held wall.

In [14]:
pin = DirichletCondition(Facet.low(i), 0)
own_rule = DerivativeCondition(Box({i: (0, 2)}), -u[i])

state_first = OrdinaryDifferentialSystem.from_sympy(
    diffusion, u, (i,), conditions=[pin, own_rule]
)
derivative_first = OrdinaryDifferentialSystem.from_sympy(
    diffusion, u, (i,), conditions=[own_rule, pin]
)

for label, ode in (("state first", state_first), ("derivative first", derivative_first)):
    held = ode.equation.expr.subs(i, 0)
    inside = ode.equation.expr.subs(i, 1)
    print(f"{label:<17} du/dt at i = 0 -> {held}   at i = 1 -> {inside}")

state first       du/dt at i = 0 -> 0   at i = 1 -> -u[1]
derivative first  du/dt at i = 0 -> 0   at i = 1 -> -u[1]


### Advanced - A held site inside a subdomain

The two families meet in one more place. When a subdomain covers a wall that also carries a state
condition, the held site is governed by the subdomain, so the branch derived for it has to use the
subdomain's stencil rather than the ambient one.

Below, plain diffusion on the ring, a box $0 \le i \le 3$ diffusing $D$ times faster, and a
mirror $u[0] = -u[1]$ on the low wall that the box covers. With `eliminate=True`, `u[0]` is folded
into the rows referencing it.

In [15]:
meeting = OrdinaryDifferentialSystem.from_sympy(
    diffusion,
    u,
    (i,),
    conditions=[
        DirichletCondition(Facet.low(i), -u[1]),  # mirror u[0] = -u[1]
        DerivativeCondition(Box({i: (0, 3)}), scaled),
    ],
    eliminate=True,
)
meeting.equation.expr

Piecewise((-D*(-3*u[1] + u[2])/h**2, Eq(i, 0)), (D*(-3*u[1] + u[2])/h**2, Eq(i, 1)), ((-u[1] + u[8] - 2*u[9])/h**2, Eq(i, 9)), (D*(u[Mod(i + 1, 10)] + u[Mod(i + 9, 10)] - 2*u[i])/h**2, i <= 3), ((u[Mod(i + 1, 10)] + u[Mod(i + 9, 10)] - 2*u[i])/h**2, True))

Reading the four regions, note where $D$ appears and where it does not:

| site | region | carries $D$ |
| --- | --- | --- |
| $i = N - 1$ | the high wall, outside the box | no, ambient diffusion, with the wrapped `u[0]` folded in as `-u[1]` |
| $i = 0$ | the mirrored wall, inside the box | yes, the mirror chain-rules through the subdomain |
| $i = 3$ | inside the box, at its interior edge | yes, reading the ambient `u[4]` outside |
| $i = 4$ | outside the box, beside that edge | no, plain ambient diffusion |

In [16]:
for site in (N - 1, 0, 3, 4):
    print(f"at i = {site:<3}", meeting.equation.expr.subs(i, site))

at i = 9   (-u[1] + u[8] - 2*u[9])/h**2
at i = 0   -D*(-3*u[1] + u[2])/h**2
at i = 3   D*(u[2] - 2*u[3] + u[4])/h**2
at i = 4   (u[3] - 2*u[4] + u[5])/h**2


The check the section is built around: the eliminated `u[0]` has to be substituted into the
*subdomain* equation, not the ambient one. At $i = 1$, inside the box and next to the held wall,
the row is the scaled stencil with $u[0] = -u[1]$. And the boundary stays consistent with its
condition: $u[0] = -u[1]$ implies $du[0]/dt = -du[1]/dt$, branch for branch.

In [17]:
row_1 = meeting.equation.expr.subs(i, 1)
expected = scaled.subs(i, 1).subs(u[0], -u[1])
print("assembled at i = 1      ", row_1)
print("scaled, u[0] = -u[1]    ", sp.expand(expected))
print("match                   ", sp.simplify(row_1 - expected) == 0)
print("keeps the coefficient D ", row_1.has(D))
print(
    "mirror stays invariant  ",
    sp.simplify(
        meeting.equation.expr.subs(i, 0) + meeting.equation.expr.subs(i, 1)
    )
    == 0,
)

assembled at i = 1       D*(-3*u[1] + u[2])/h**2
scaled, u[0] = -u[1]     -3*D*u[1]/h**2 + D*u[2]/h**2
match                    True
keeps the coefficient D  True
mirror stays invariant   True


## The same, from strings

`from_strings` is this assembly behind a parser. The interior is one string; the conditions are
passed as objects rather than encoded in it.

In [18]:
parsed = OrdinaryDifferentialSystem.from_strings(
    "(u[i-1] + u[i+1])/h",
    indices=(i,),
    conditions=[
        DirichletCondition(Facet.low(i), 0),
        DirichletCondition(Facet.high(i), 0),
    ],
)
parsed.equation.expr

Piecewise((0, Eq(i, 0) | Eq(i, 9)), ((u[Mod(i + 1, 10)] + u[Mod(i + 9, 10)])/h, True))

An edge that wraps needs no condition at all, because wrapping is the default.

---

[← 5. Vectors](06_vector.ipynb) · [Notebook index](../README.md) · [End to end →](08_end_to_end.ipynb)